# E1.1 · Detumble data (practice)

**Phase E Operations and Sustainment › Attitude Control (ADCS) › E1 In-orbit detumble** · PM card TC-ADCS-E1

> 💡 This is much easier after you've followed **A2 B-dot study + sim**.

You've received the gyro (angular rate sensor) data from right after separation. Work out how fast the satellite is spinning, and around which axis.

- Fill in the places marked `# TODO` and run them.
- Run the **check cell** right below to see whether you got it.

## Setup: receive the data (just run this)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Spacecraft: FASTSAT (B-dot paper, section V)
J = np.diag([15.9, 16.9, 11.7])   # moment of inertia [kg·m²]
J_INV = np.linalg.inv(J)
M_MAX = 5.0                        # max magnetorquer dipole moment [A·m²]

# Orbit: 650 km altitude, 72° inclination
MU_E = 3.986e14                    # Earth's gravitational parameter [m³/s²]
R = 6371e3 + 650e3                 # orbit radius [m]
INC = np.radians(72)
T_ORBIT = 2 * np.pi * np.sqrt(R**3 / MU_E)   # orbital period [s]
n = 2 * np.pi / T_ORBIT                      # mean motion [rad/s]
MU_F = 7.95e15                     # Earth's magnetic moment constant [T·m³]

print(f"Orbital period: {T_ORBIT / 60:.1f} min")

def earth_field(t):
    """Earth's magnetic field in the orbit frame [T]: dipole model, paper eq. (1)"""
    return MU_F / R**3 * np.array([
        np.cos(n * t) * np.sin(INC),
        -np.cos(INC),
        2 * np.sin(n * t) * np.sin(INC),
    ])

def skew(w):
    """Cross product as a matrix: skew(a) @ b == np.cross(a, b)"""
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])

def rotate(C, w, dt):
    """Update attitude matrix C after spinning at rate w for dt seconds (Rodrigues' formula)"""
    rate = np.linalg.norm(w)
    if rate < 1e-12:
        return C
    K = skew(w / rate)
    th = rate * dt
    return (np.eye(3) + np.sin(th) * K + (1 - np.cos(th)) * K @ K).T @ C

def simulate(controller, orbits=3, dt=1.0, w0_deg=(1, 3, 1)):
    """Simulate the spacecraft spin with controller(b_now, b_prev, dt) -> dipole moment m.
    Returns (orbits, angular rate [deg/s])."""
    steps = int(orbits * T_ORBIT / dt)
    w = np.radians(w0_deg).astype(float)   # tip-off rate after separation (paper: pitch 3°/s, roll and yaw 1°/s)
    C = np.eye(3)                           # starting attitude
    b_prev = C @ earth_field(0)
    ts, ws = np.zeros(steps), np.zeros((steps, 3))
    for i in range(steps):
        t = i * dt
        b = C @ earth_field(t)              # magnetometer reading (body frame)
        m = controller(b, b_prev, dt)       # dipole moment chosen by the controller
        tau = np.cross(m, b)                # magnetic torque τ = m × B
        w = w + J_INV @ (tau - np.cross(w, J @ w)) * dt   # Euler's rotation equation
        C = rotate(C, w, dt)
        b_prev = b
        ts[i], ws[i] = t, w
    return ts / T_ORBIT, np.degrees(ws)

def no_control(b_now, b_prev, dt):
    return np.zeros(3)

def check(ok, hint):
    if ok:
        print("✅ Passed")
    else:
        print("❌ Not yet. " + hint)

# Gyro data for the first half orbit after separation (with sensor noise)
rng = np.random.default_rng(0)
_orbits, _rates = simulate(no_control, orbits=0.5, dt=2.0)
t_min = _orbits * T_ORBIT / 60                                       # time [min]
gyro_rad = np.radians(_rates) + rng.normal(0, 0.002, _rates.shape)  # gyro output [rad/s]
print("Data shape:", gyro_rad.shape, "(time, axis x·y·z)")

## Problem 1. Change units
The gyro reports **rad/s**. Convert it to **deg/s**, which is easier to read.

Hint: `np.degrees()`

In [ ]:
gyro_deg = None  # TODO

In [ ]:
# Check
check(gyro_deg is not None and np.allclose(gyro_deg, np.degrees(gyro_rad)),
      "Convert all of gyro_rad to deg/s.")

## Problem 2. Total spin rate
Find the total spin rate |ω| = √(ωx² + ωy² + ωz²) at each time step.

Hint: `np.linalg.norm(..., axis=1)`

In [ ]:
rate = None  # TODO (in deg/s)

In [ ]:
# Check
check(rate is not None and np.shape(rate) == (len(gyro_deg),)
      and np.allclose(rate, np.linalg.norm(np.degrees(gyro_rad), axis=1)),
      "You need one value per time step, in deg/s.")

## Problem 3. Fastest axis
Which axis has the largest average spin rate (absolute value)? Answer `0` (x), `1` (y) or `2` (z).

Hint: `np.abs`, `.mean(axis=0)`, `np.argmax`

In [ ]:
fastest_axis = None  # TODO

In [ ]:
# Check
check(fastest_axis == int(np.argmax(np.abs(np.degrees(gyro_rad)).mean(axis=0))),
      "Average the absolute value per axis and pick the index of the largest.")

## Problem 4. Plot it (no check)
Plot the three axes and |ω| against `t_min` on one chart. `plot_rates` from A2 is a good reference.

In [ ]:
plt.figure(figsize=(9, 3.5))
# TODO: plot x, y, z and |ω| with plt.plot(...)

plt.xlabel("time [min]")
plt.ylabel("angular rate [deg/s]")
plt.grid(alpha=0.3)
plt.show()

### Next →
In **E1.2 Compare vs sim**, compare this data with your A2 simulation.